<h1>3.3 DSSD 相邻条关联</h1>
<h2>相邻条信号的物理来源</h2>
<p>粒子入射到两条之间或邻近区域时，产生的电荷可能由两条共同收集，形成 charge sharing。一个较强的局部信号也可能在邻条产生相关响应。即使沉积能量相同，两条信号的分配仍会随入射位置和电荷收集条件改变。</p>
<p>下图比较了不同入射位置。低能 α 粒子停止在探测器表面附近时，条间区 B–F 的响应较复杂；例如 B、F 位置可以出现异极性信号。高能粒子穿过探测器时，图中 A、G、H 一类轨迹可在相应电极形成同极性的信号。这说明“相邻两条都有信号”并不自动等于一个简单的能量加和模型。</p>
<img alt="" src="fig/dssd-n3.png"/>
<p>本例电子学记录所选极性的脉冲幅度。对于两条均有有效信号、电荷收集完整的 sharing 事例，刻度后的能量近似满足</p>
<p>$$E=E_i+E_{i+1}.$$</p>
<p>结合另一面的能量检验后，可将这两条组成一个 cluster，保留其总能量。相邻关系是候选条件，能量一致性才进一步支持这种解释。</p>
<h2>实验数据</h2><p>S4 的 Ring 条较窄、条间距离小，相邻条关联比 Pie 面更明显。下面以 Ring 面为主，依次观察关联、选择参考条带、求取刻度参数，再重建能量和位置。</p>

In [1]:
%jsroot on

In [2]:
//%jsroot on
TFile *fin = new TFile("s4hit.root");
TTree *tree = (TTree*)fin->Get("tree");
TCanvas *c1 = new TCanvas("c1","c1",800,400);
c1->SetLogz();

<h2>相邻条的幅度关联</h2><p>以第 13、14 条为例，Ring 面出现清楚的负斜率条带：同一粒子在一条上分配的信号增大，另一条就减小。后面用已知 α 能量检验两条的加和。Pie 面相同条件下的结构弱得多。</p>

In [3]:
c1 = new TCanvas("cAdjacent","cAdjacent",800,400);
tree->Draw("re[14]:re[13]>>hr43(800,-100,1700,800,-100,1700)","","goff");//goff 选项：只填充hist，而不显示。
tree->Draw("pe[14]:pe[13]>>hp43(800,-100,1700,800,-100,1700)","","goff");
c1->Divide(2,1);
c1->cd(1);      
hr43->Draw("colz");
c1->cd(2);       
hp43->Draw("colz");
c1->Draw();

<h2>相邻条信号对单条能谱的影响</h2><p>红谱选择邻条有信号的事例，黑谱选择两侧邻条均无有效信号的事例。sharing 使单条只记录总能量的一部分，因此原本集中的全能峰向较低幅度延伸。红谱富集 sharing，也可能包含串扰和偶然符合。</p><p>这正是单条刻度与相邻条重建需要分别处理的原因：先用合适的单条样本刻度，再判断共享事例是否能够加和恢复。</p>

In [4]:
c1 = new TCanvas("cSharing","cSharing",800,400);
c1->Divide(1,1);
c1->cd();
tree->Draw("re[13]>>h1a(800,0,1600)","re[14]>0 || re[12]>0");// 有相邻条能量共享
TH1F *h1a = (TH1F*)gROOT->FindObject("h1a");
tree->Draw("re[13]>>h2a(800,0,1600)","re[14]<0 && re[12]<0");// 没有相邻条能量共享
h1a->SetLineColor(kRed);
h1a->Draw("same");
c1->Draw();

<h2>选择两条已知能量的参考带</h2><p>下面调用已经保存的图形 cut：<code>cut1</code> 对应 8.6931 MeV 的 α 线，<code>cut2</code> 对应 6.6708 MeV 的 α 线。两条带中的粒子能量已知，但每个事例在两条之间的分配不同，可用来约束两条的响应。这里直接应用图形 cut，绘制方法见 ROOT 基础教程。</p>

In [5]:
.x cut1.C

In [6]:
.x cut2.C

In [7]:
c1 = new TCanvas("cBandSelection","cBandSelection",800,400);
c1->Clear();
c1->Divide(2,1);
c1->cd(1);      
tree->Draw("re[14]:re[13]>>hcut1(800,0,1600,800,0,1600)","cut1","colz");
c1->cd(2);
tree->Draw("re[14]:re[13]>>hcut2(800,0,1600,800,0,1600)","cut2","colz");
c1->Draw();

<h2>能量共享与线性刻度</h2>
<p>用 $A_i,A_{i+1}$ 表示两条的原始幅度，分别采用线性刻度：</p>
<p>$$E_i=k_iA_i+b_i,\qquad E_{i+1}=k_{i+1}A_{i+1}+b_{i+1}.$$</p>
<p>电荷完整收集时，同一粒子的总能量为</p>
<p>$$E=k_iA_i+k_{i+1}A_{i+1}+(b_i+b_{i+1}).$$</p>
<p>把第二条的幅度写成第一条的函数，得到</p>
<p>$$A_{i+1}=K A_i+B,\qquad K=-\frac{k_i}{k_{i+1}},\qquad B=\frac{E-(b_i+b_{i+1})}{k_{i+1}}.$$</p>
<p>因而不同 α 能量对应一组近似平行的直线。两条已知能量 $E_a,E_b$ 给出的截距分别为 $B_a,B_b$，则</p>
<p>$$k_{i+1}=\frac{E_a-E_b}{B_a-B_b},\qquad k_i=-Kk_{i+1},\qquad b_i+b_{i+1}=E_a-k_{i+1}B_a.$$</p>
<p>两条线的斜率在误差内一致时，可以估计共同斜率 $K$。这里能确定两个增益及 offset 之和，不能仅凭这两条带分别确定 $b_i$ 和 $b_{i+1}$。</p>

<h2>对参考带作直线拟合</h2><p>二维直方图用于观察分布。拟合时，用 <code>TTree::Draw</code> 的 <code>goff</code> 选出逐事例数值，再构造 <code>TGraph</code>：</p><pre><code>tree::Draw("x1:x2",selection);
TGraph *gr = new TGraph(tree-&gt;GetSelectedRows(), tree-&gt;GetV2(), tree-&gt;GetV1());
</code></pre>
<p><code>GetV1()</code>、<code>GetV2()</code> 按 Draw 表达式的顺序返回数组。表达式为 <code>y:x</code> 时，构造 TGraph 的横坐标用 GetV2，纵坐标用 GetV1。先用 <code>SetEstimate</code> 设置足够的缓冲容量；TGraph 构造时复制这些数据，后续 Draw 会刷新原缓冲区。</p>

In [8]:
TGraph *gr1, *gr2;
TF1 *f1, *f2;

In [9]:
c1 = new TCanvas("cBandFits","cBandFits",800,400);
tree->SetEstimate(tree->GetEntries()+1);
c1->Clear();
c1->Divide(2,1);
c1->cd(1); 

tree->Draw("re[14]:re[13]","cut1","");
gr1 = new TGraph(tree->GetSelectedRows(), tree->GetV2(), tree->GetV1());
gr1->Fit("pol1","Q");
f1 = gr1->GetFunction("pol1");
f1->Draw("same");

c1->cd(2);
tree->Draw("re[14]:re[13]","cut2","");
gr2 = new TGraph(tree->GetSelectedRows(), tree->GetV2(), tree->GetV1());
gr2->Fit("pol1","Q");
f2 = gr2->GetFunction("pol1");
f2->Draw("same");
c1->Draw();

<h2>从拟合参数求刻度系数</h2><p>下面先将两条线的斜率作等权平均，作为共同斜率的初步估计，再代入上式计算 $k_{13}$、$k_{14}$ 和 $b_{13}+b_{14}$。打印两个斜率是为了检查两条带是否接近平行；若差异超过拟合不确定度，应回到 cut、能量收集及模型检查，而不是用平均掩盖差异。</p>

In [10]:
double E1 = 8.6931;
double E2 = 6.6708;
double K13,K14,K,B13,B14,k13,k14,b13_14;

K13 = f1->GetParameter(1);
K14 = f2->GetParameter(1);
K  = (K13 + K14)/2.0;

B13 = f1->GetParameter(0);
B14 = f2->GetParameter(0);

k14  = (E1-E2)/(B13-B14);
k13  = -K*k14;
b13_14 = E1 - k14*B13;

cout << Form("k13 = %f, k14 = %f, b13 + b14 = %f",k13,k14,b13_14) << endl;
cout << "Band slopes: " << K13 << " +/- " << f1->GetParError(1)
     << ", " << K14 << " +/- " << f2->GetParError(1) << endl;

k13 = 0.007014, k14 = 0.006918, b13 + b14 = -0.826077
Band slopes: -1.01359 +/- 0.000281509, -1.01424 +/- 0.000274002


<h2>重建共享事例的总能量</h2><p>逐事例按两条的增益和 offset 之和计算总能量。原来的斜条带随后变成一维能量谱中的峰。这样，满足共享模型的事例能够保留在后续分析中，而不必因为单条没有记录全部能量就全部舍弃。</p><p>这里的直线回归用于初步刻度。两条的幅度都有测量涨落，而普通 TGraph 的 least squares 只处理纵向残差；若需精确的参数误差，应采用两轴误差模型，并传播 slope 与 intercept 的 covariance。</p>

In [11]:
c1 = new TCanvas("cEnergySum","cEnergySum",800,400);
tree->SetAlias("e13_14",Form("%f*re[13] + %f*re[14] + %f",k13,k14,b13_14));
tree->Draw("e13_14>>hsum(2000,0,10)","re[13]>0 && re[14]>0");
c1->Draw();

<h2>相邻条事例的位置估计</h2><p>相邻条同时有信号不表示粒子恰好打在几何间隙。入射位置、入射角、电荷共享及电子学耦合都可能改变信号比例。用于位置细化时，先选择恰有两条相邻信号、两信号均高于阈值、附近没有额外条参与、能量和与一个粒子相符的样本。</p><p>设条中心位置为 $x_i,x_{i+1}$，一个简单估计量是能量重心：</p>
<p>$$x_c=\frac{E_ix_i+E_{i+1}x_{i+1}}{E_i+E_{i+1}}.$$</p>
<p>它给出条间连续的位置估计，比只取较大信号的条号包含更多信息。但连续坐标不等于已经获得相应的位置分辨率。需要用已知位置的扫描或独立径迹标定“电荷比—入射位置”关系，再检查阈值、损失电荷和串扰带来的偏移。</p>